# Value-Guided Search — Colab pipeline

**Goal:** test whether a *learned value function* guiding the search beats the heuristic — the
question step 3 left open. Step 3 used a heuristic-rollout ISMCTS and lost (11–17% vs heuristic).
This uses `value_trees.json` (win-prob value net, AUC 0.976) to score each candidate move's
resulting state directly, with the improved archetype-prior determinization. Then, if promising,
an AlphaZero-style self-play loop sharpens the value model and re-gates.

**Runs natively on Colab** (linux/amd64 — the engine `libcg.so` needs no emulation here).

### Hardware note
The value model is **XGBoost trees scored in NumPy on CPU**, and self-play runs the **CPU-bound**
C engine. **A GPU (A100 or otherwise) does almost nothing for this pipeline** — throughput is gated
by CPU cores (parallel self-play games), not GPU. Pick a **high-CPU** runtime; a GPU only matters
if you later swap the tree value-model for a neural net (a possible *next* experiment, not this one).

### What you need to provide
- **The repo** (via `git clone` below) — brings all code + the small weights (`value_trees.json`, ~0.6 MB).
- **Nothing on Drive is required.** Self-play generates its own training data on the engine.
- *(Optional)* `replays.sqlite` (3.3 GB) on Drive **only** if you want to retrain from historical
  replays instead of self-play — the notebook detects it and skips those cells if absent.

### Output
- `value_search_result.json` — win-rate of value-guided search vs the heuristic (the verdict).
- `selfplay.npz` + a retrained `card_policy.npz` **only if** the self-play gate passes.
- `value_search_gate.png` — the summary figure.


## 1 · Clone the repo & verify the native engine


In [ ]:
# --- Clone the repo. Works for a PUBLIC repo (just Enter at the prompt) or a
# PRIVATE repo (paste a fine-grained token with read-only 'Contents' access). ---
REPO_OWNER = 'cha7ura'
REPO_NAME  = 'kaggle-pokemon-tcg'
BRANCH     = 'feat/imitation-league'

import os, subprocess, sys, getpass
if not os.path.isdir('/content/repo'):
    tok = getpass.getpass('GitHub token (PUBLIC repo? just press Enter): ').strip()
    auth = f'{tok}@' if tok else ''            # empty -> anonymous (public) clone
    url  = f'https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git'
    r = subprocess.run(['git','clone','--depth','1','-b',BRANCH,url,'/content/repo'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(tok,'***') if tok else r.stderr   # scrub token from errors
        print(msg)
        raise SystemExit('clone failed — see the git error above (check branch name; if private, the token needs read access)')
    print('cloned OK')
os.chdir('/content/repo')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.'))[:20])


In [ ]:
# Verify the engine is the native x86 shared lib and it loads.
# NOTE: sdk/cg/libcg.so is gitignored (`.gitignore`: sdk/cg/*.so) so the clone does NOT bring it.
# Provide the prebuilt Linux x86_64 binary once via Drive (MyDrive/libcg.so); this copies it in.
import os, shutil, subprocess, hashlib
if not os.path.exists('sdk/cg/libcg.so'):
    src = None
    for cand in ('/content/drive/MyDrive/libcg.so', 'libcg.so'):
        if os.path.exists(cand): src = cand; break
    if src is None:
        try:
            from google.colab import drive; drive.mount('/content/drive')
            if os.path.exists('/content/drive/MyDrive/libcg.so'): src = '/content/drive/MyDrive/libcg.so'
        except Exception as e: print('drive not mounted:', e)
    if src is None:
        raise SystemExit('libcg.so missing. Upload the Linux x86_64 build to MyDrive/libcg.so '
                         '(it is gitignored, so not in the clone), then re-run this cell.')
    shutil.copy(src, 'sdk/cg/libcg.so')
print('libcg.so md5:', hashlib.md5(open('sdk/cg/libcg.so','rb').read()).hexdigest())

import platform
print('platform:', platform.platform(), '| machine:', platform.machine())   # expect x86_64 Linux
print(subprocess.run(['file','sdk/cg/libcg.so'], capture_output=True, text=True).stdout.strip())

import sys; sys.path.insert(0,'sdk'); sys.path.insert(0,'autoresearch')
from cg.game import battle_start, battle_select, battle_finish   # dlopen libcg.so happens here
print('engine loaded OK')

In [ ]:
# Smoke test: 3 random self-play games (sanity — same as sdk/smoke_test.py).
# smoke_test.py opens 'deck.csv' relative to cwd, which lives in sdk/, so run from there.
import subprocess
r = subprocess.run([sys.executable,'smoke_test.py','3'], capture_output=True, text=True,
                   cwd='sdk', env={**os.environ,'PYTHONPATH':'.'})
print(r.stdout or r.stderr[-1500:])

## 2 · Build the value-guided search agent

The repo ships `autoresearch/agent_vsearch.py` — a value-guided 1-ply search that scores each
candidate move's resulting state with the learned win-prob net. We adapt it two ways for a fair
test: **(a)** pilot the same `lucario_meta` deck step 3 used (so the number is directly comparable),
and **(b)** use the improved archetype-prior determinization from `determinization/determinize.py`
instead of the weak basic-energy filler. The agent below writes itself into the repo as
`agent_vsearch_prior.py`.


In [ ]:
VSEARCH_PRIOR = r'''
"""Value-guided 1-ply search with archetype-prior determinization, deck-agnostic base pilot.
Scores each candidate MAIN move by forward-simming one engine step and evaluating the resulting
state with the learned win-prob value net (value_trees.json). Falls back to the heuristic pilot
for non-MAIN / sub-selection decisions. Crash-safe."""
import os, json, math
from cg.api import (to_observation_class, OptionType, SelectType,
                    search_begin, search_step, search_end, all_card_data, all_attack)
import agent_lucario as H              # base pilot + deck (same as the step-3 gate)

NE, SF = 12, 6 + 12 + 2 + 4
ENERGY_ID = 3
CD = {c.cardId: c for c in all_card_data()}
ATK = {a.attackId: a for a in all_attack()}

# --- optional archetype-prior determinizer (falls back to basic-energy if module absent) ---
_DET = None
try:
    import sys as _sys
    for _p in ('determinization', '.', 'autoresearch'):
        if os.path.isdir(_p) and _p not in _sys.path: _sys.path.insert(0, _p)
    from determinize import Determinizer
    for _pr, _cf in (('determinization/arch_priors.json','determinization/cards_full.csv'),
                     ('arch_priors.json','cards_full.csv'),
                     ('autoresearch/arch_priors.json','autoresearch/cards_full.csv')):
        if os.path.exists(_pr) and os.path.exists(_cf):
            _DET = Determinizer.load(_pr, _cf); break
except Exception as _e:
    _DET = None

def _oh(i, n):
    v = [0.0]*n
    if i is not None and 0 <= int(i) < n: v[int(i)] = 1.0
    return v
def _cardfeat(cid):
    c = CD.get(cid)
    if not c: return [0.0]*SF
    bd = max([ATK[a].damage for a in c.attacks if a in ATK], default=0)
    return (_oh(int(c.cardType),6)+_oh(int(c.energyType),NE)
            +[1.0 if c.ex else 0.0,1.0 if c.megaEx else 0.0]
            +[c.hp/300, c.retreatCost/4, len(c.attacks)/4, bd/300])
CF0 = [0.0]*SF
def _mons(p):
    return (([p.active[0]] if p.active and p.active[0] else [])+[b for b in p.bench if b])[:6]
def _context(st):
    me = st.yourIndex; my = st.players[me]; op = st.players[1-me]
    def pool(p):
        ms=_mons(p)
        if not ms: return CF0+[0.0,0.0]
        cols=list(zip(*[_cardfeat(m.id) for m in ms]))
        cf=[sum(c)/len(ms) for c in cols]
        return cf+[sum(m.hp/300 for m in ms)/len(ms), sum(len(m.energies)/4 for m in ms)/len(ms)]
    state=[len(my.prize)/6,len(op.prize)/6,len(my.bench)/5,len(op.bench)/5,
           len(my.hand or [])/10, st.turn/40,
           float(my.poisoned),float(my.burned),float(my.asleep),float(my.paralyzed),float(my.confused)]
    return pool(my)+pool(op)+state
def _load(p_):
    for p in (p_, os.path.join(os.path.dirname(__file__),p_), '/kaggle_simulations/agent/'+p_):
        if os.path.exists(p):
            d=json.load(open(p)); return d['trees'], d['base']
    return None, 0.0
_VT,_VB = _load('value_trees.json')
def _tscore(node,f):
    while 'leaf' not in node:
        nxt = node['yes'] if f[int(node['split'][1:])] < node['split_condition'] else node['no']
        node = next(c for c in node['children'] if c['nodeid']==nxt)
    return node['leaf']
def _value(st, me):
    raw = _VB + sum(_tscore(t,_context(st)) for t in _VT)
    v = 1/(1+math.exp(-raw))
    return v if st.yourIndex==me else 1-v

def _revealed_opp(state, me):
    opp = state.players[1-me]; seen=[]
    try:
        for m in _mons(opp): seen.append(m.id)
        for c in (opp.discard or []): seen.append(getattr(c,'id',c))
    except Exception: pass
    return [c for c in seen if isinstance(c,int)]

def _determinize(state, rng=None):
    me = state.yourIndex; my, opp = state.players[me], state.players[1-me]
    oa = []
    if opp.active and opp.active[0] is None: oa=[_H_basic()]
    if _DET is not None:
        try:
            import random as _r; rng = rng or _r.Random(0)
            rev = _revealed_opp(state, me)
            arch = _DET.infer_archetype(rev)
            opp_deck = _DET.sample_hidden(rev, opp.deckCount, arch, rng, stochastic=True)
            opp_hand = _DET.sample_hidden(rev, opp.handCount, arch, rng, stochastic=True)
            if len(opp_deck)>=opp.deckCount and len(opp_hand)>=opp.handCount:
                return (H._DECK[:], [ENERGY_ID]*len(my.prize), opp_deck[:opp.deckCount],
                        [ENERGY_ID]*len(opp.prize), opp_hand[:opp.handCount], oa)
        except Exception: pass
    # fallback: basic-energy filler (the step-3 baseline)
    return (H._DECK[:], [ENERGY_ID]*len(my.prize), H._DECK[:],
            [ENERGY_ID]*len(opp.prize), [ENERGY_ID]*opp.handCount, oa)
def _H_basic():
    try: return H._DECK[0]
    except Exception: return ENERGY_ID

def _eval_action(obs, action, me):
    ss = search_begin(obs, *_determinize(obs.current)); sid = ss.searchId
    try:
        ss = search_step(sid, action); cur = ss.observation.current
        if cur is None: return 0.0
        if cur.result == me: return 1.0
        if cur.result == (1-me): return 0.0
        return _value(cur, me)
    finally:
        search_end()

def agent(obs_dict):
    try:
        obs = to_observation_class(obs_dict)
        if obs.select is None: return H._DECK
        sel = obs.select
        cands = [i for i in range(len(sel.option)) if int(sel.option[i].type)!=int(OptionType.END)]
        if (_VT and int(sel.type)==int(SelectType.MAIN) and sel.maxCount==1 and sel.minCount==1
                and len(cands)>=2):
            me = obs.current.yourIndex; best_i,best_v = None,-1.0
            for i in cands:
                try: v = _eval_action(obs,[i],me)
                except Exception: v = -1.0
                if v>best_v: best_v,best_i = v,i
            if best_i is not None: return [best_i]
        return H.select_indices(obs) if hasattr(H,'select_indices') else H.agent(obs_dict)
    except Exception:
        try: return H.agent(obs_dict)
        except Exception: return [0]
'''
open('autoresearch/agent_vsearch_prior.py','w').write(VSEARCH_PRIOR)
print('wrote autoresearch/agent_vsearch_prior.py (%d bytes)' % len(VSEARCH_PRIOR))


## 3 · Gate — value-guided search vs the heuristic

The decisive test. Same controlled setup as the step-3 gate: **both seats play `lucario_meta`**,
seat alternates each game, only the pilot differs (value-search vs heuristic). This is directly
comparable to step 3's 11–17%.

**Read the result (single threshold, used everywhere in this notebook):**
- **≥ 55%** → clear win; a learned value function fixes what heuristic-rollout search could not → proceed to self-play (§4).
- **50–55%** → marginal; an edge may exist but isn't decisive at 100 games (widen to 200+ before trusting it).
- **< 50%** (esp. teens, like step 3) → search does not beat the heuristic regardless of the evaluator; the search question is closed.


In [ ]:
import os
os.environ['AGENT_DECK'] = 'autoresearch/decks/lucario_meta.csv'   # agent_lucario resolves this rel to cwd
import random, json, time
from cg.game import battle_start, battle_select, battle_finish
import agent_lucario as HEUR
import importlib, agent_vsearch_prior as VS; importlib.reload(VS)
from cg.api import SelectType

DECK = [int(l) for l in open('autoresearch/decks/lucario_meta.csv') if l.strip()][:60]
GAMES = 100        # bump to 200+ for tighter CI once you trust it

# The designed test is value-net + ARCHETYPE-PRIOR determinization. If the prior module didn't
# load (determinize.py / arch_priors.json untracked in the clone), the agent silently falls back
# to basic-energy filler and the run tests a DIFFERENT, weaker config. Make that loud, not silent.
det = 'archetype-prior' if VS._DET is not None else 'basic-energy fallback'
print('value net loaded:', VS._VT is not None, '| determinization:', det)
if VS._DET is None:
    print('\n' + '='*68)
    print('!! WARNING: prior-determinizer NOT loaded -> BASIC-ENERGY FALLBACK.')
    print('   This is NOT the designed value-net + prior config. To fix, ensure')
    print('   determinize.py + arch_priors.json are in the clone (commit them;')
    print('   they load alongside autoresearch/cards_full.csv). The result below')
    print('   is labelled "basic-energy fallback" so it cannot be mis-cited.')
    print('='*68 + '\n')

def play(vs_seat, max_steps=20000):
    obs, sd = battle_start(DECK, DECK)
    if obs is None: return None
    steps = 0
    try:
        while True:
            cur = obs['current']
            if cur is not None and cur.get('result',-1)!=-1: return cur['result']
            if obs['select'] is None: return None
            seat = cur['yourIndex']
            action = VS.agent(obs) if seat==vs_seat else HEUR.agent(obs)
            obs = battle_select(action); steps += 1
            if steps>max_steps: return None
    finally:
        battle_finish()

random.seed(0); vw=hw=draw=err=0; per=[]; seat={0:[0,0],1:[0,0]}
t0=time.time()
for g in range(GAMES):
    s=g%2; r=play(s); seat[s][1]+=1
    if r==s: vw+=1; seat[s][0]+=1
    elif r==(1-s): hw+=1
    elif r==2: draw+=1
    else: err+=1
    per.append({'game':g,'vs_seat':s,'result':r})
    if (g+1)%10==0:
        dec=vw+hw; wr=100*vw/dec if dec else 0
        print(f'  {g+1:4}/{GAMES}  vsearch {vw}  heur {hw}  draw {draw}  err {err}  | vsearch WR {wr:.1f}%  ({time.time()-t0:.0f}s)')

dec=vw+hw; wr=100*vw/dec if dec else 0
print(f'\nVALUE-SEARCH win-rate vs heuristic: {wr:.1f}%  ({vw}/{dec})')
print(f'as seat0: {seat[0]}  as seat1: {seat[1]}   (check no seat artifact)')
print(f'determinization: {det}')
json.dump({'winrate_pct':round(wr,1),'vsearch_wins':vw,'heur_wins':hw,'draws':draw,'errors':err,
           'games':GAMES,'seatwins':seat,'determinization':det,'per_game':per},
          open('value_search_result.json','w'), indent=2)
print('wrote value_search_result.json')

## 4 · Self-play sharpening loop  *(run only if §3 is promising, ≳52%)*

AlphaZero-style: the value-search agent plays itself with temperature exploration across the field
decks; winners' decisions become new training rows. Then a **gated retrain** — the augmented model
is kept **only if** it beats the baseline on held-out *real human* decisions (the anti-sim-overfit
gate the old RL skipped). Uses the repo's own `selfplay.py` + `selfplay_retrain.py`.

CPU-bound; scale `--games` with your core count. This is where more CPUs (not GPU) help.


In [ ]:
# 4a · generate self-play winner-rows (writes tools/imitation/data/selfplay.npz)
import subprocess, sys, os
os.makedirs('tools/imitation/data', exist_ok=True)
SELFPLAY_GAMES = 200   # raise with CPU cores available
r = subprocess.run([sys.executable,'tools/imitation/selfplay.py','--games',str(SELFPLAY_GAMES),
                    '--temp','1.0','--out','tools/imitation/data/selfplay.npz'],
                   env={**os.environ,'PYTHONPATH':'sdk:.'}, capture_output=True, text=True)
print(r.stdout[-2000:]); print('ERR', r.stderr[-800:] if r.returncode else 'ok')


In [ ]:
# 4b · gated retrain — keep augmented model ONLY if it beats baseline on held-out REAL decisions
# Needs the imitation decisions set (tools/imitation/data/decisions.npz). If absent, this reports so.
import os, subprocess, sys
if not os.path.exists('tools/imitation/data/decisions.npz'):
    print('decisions.npz not found — the imitation training set is not in the repo.')
    print('Self-play rows are in selfplay.npz; to run the gate, add decisions.npz (rebuild via')
    print('tools/imitation/build_decisions.py from replays.sqlite, or copy from a prior run).')
else:
    r = subprocess.run([sys.executable,'-m','tools.imitation.selfplay_retrain'],
                       env={**os.environ,'PYTHONPATH':'sdk:.'}, capture_output=True, text=True)
    print(r.stdout[-2000:]); print('ERR', r.stderr[-800:] if r.returncode else 'ok')


## 5 · Optional — retrain the VALUE net from historical replays

Only if you uploaded `replays.sqlite` to Drive. Mounts Drive, points the value-net trainer at it.
Skipped automatically if the file is absent. (Self-play in §4 does not need this.)


In [ ]:
import os
SQLITE = None
for cand in ('/content/drive/MyDrive/replays.sqlite','replays.sqlite'):
    if os.path.exists(cand): SQLITE = cand; break
if SQLITE is None:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        if os.path.exists('/content/drive/MyDrive/replays.sqlite'):
            SQLITE='/content/drive/MyDrive/replays.sqlite'
    except Exception as e: print('drive not mounted:', e)
print('replays.sqlite:', SQLITE or 'ABSENT — skipping value-net retrain (self-play path in §4 is unaffected)')
# If present, the repo's value-net trainer can be pointed at SQLITE here.
# (Left as an explicit step so you control the ~hour-long rebuild.)


## 6 · Summary figure & verdict


In [ ]:
import json, matplotlib.pyplot as plt
r = json.load(open('value_search_result.json'))
wr = r['winrate_pct']
fig, ax = plt.subplots(figsize=(5.2,3.8))
color = '#2e7d32' if wr>=55 else ('#e08a2e' if wr>=50 else '#c0392b')
ax.bar(['value-search\nvs heuristic'], [wr], width=0.5, color=color, zorder=3)
ax.axhline(50, ls='--', lw=1.2, color='0.4')
ax.axhline(17, ls=':', lw=1.0, color='0.6'); ax.text(0.52, 17, 'step-3 rollout search (17%)', fontsize=7, color='0.5', va='center')
ax.text(0, wr+1.5, f'{wr:.0f}%', ha='center', fontweight='bold')
ax.set_ylim(0,80); ax.set_ylabel('win-rate (%)')
ax.set_title(f'Value-guided search: {wr:.0f}% vs heuristic  ({r["determinization"]})', fontsize=9)
fig.tight_layout(); fig.savefig('value_search_gate.png', dpi=200, bbox_inches='tight')
verdict = ('CLEAR WIN - proceed to self-play (>=55%)' if wr>=55 else
           'MARGINAL - widen to 200+ games before trusting (50-55%)' if wr>=50 else
           'search does not beat heuristic - closes the search question (<50%)')
print('verdict:', verdict)
plt.show()


---
### After the run — send back to the analysis session
Download and share these so the verdict can be written up:
- `value_search_result.json` — the win-rate + per-game log
- `value_search_gate.png` — the figure
- self-play gate stdout from §4b (the held-out real top-1 delta) if you ran it
